# BirdCLEF+ 2026 - Exploratory Data Analysis

Analyze the competition dataset: species distribution, recording properties, geographic distribution, and spectrogram visualization.

In [ ]:
import os
import sys

# Detect environment
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    KAGGLE_USER = "stochastix"
    sys.path.insert(0, f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src")
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
else:
    sys.path.insert(0, "..")
    DATA_DIR = "../data"

TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "train_audio")
TRAIN_SOUNDSCAPES_DIR = os.path.join(DATA_DIR, "train_soundscapes")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from IPython.display import Audio, display

from src.audio import load_audio, audio_to_melspec

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

print(f"ON_KAGGLE: {ON_KAGGLE}")
print(f"DATA_DIR: {DATA_DIR}")

## 1. Load Metadata

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
taxonomy_df = pd.read_csv(os.path.join(DATA_DIR, "taxonomy.csv"))

print(f"Training recordings: {len(train_df)}")
print(f"Species in taxonomy: {len(taxonomy_df)}")
print(f"Unique species in training: {train_df['primary_label'].nunique()}")

print("\n--- train.csv columns ---")
print(train_df.columns.tolist())
train_df.head()

In [ ]:
print("--- taxonomy.csv ---")
print(taxonomy_df.columns.tolist())
taxonomy_df.head(10)

## 2. Species Distribution

In [ ]:
class_counts = train_df["primary_label"].value_counts()

print(f"Most common:  {class_counts.head(5).to_dict()}")
print(f"Least common: {class_counts.tail(5).to_dict()}")
print(f"Median count: {class_counts.median():.0f}")
print(f"Imbalance ratio: {class_counts.max() / class_counts.min():.1f}x")

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Full distribution (sorted)
axes[0].bar(range(len(class_counts)), class_counts.values, width=1.0)
axes[0].set_xlabel("Species (sorted by count)")
axes[0].set_ylabel("Number of recordings")
axes[0].set_title(f"Recordings per species ({len(class_counts)} species)")

# Histogram of counts
axes[1].hist(class_counts.values, bins=50, edgecolor="black")
axes[1].set_xlabel("Number of recordings")
axes[1].set_ylabel("Number of species")
axes[1].set_title("Distribution of recording counts")

plt.tight_layout()
plt.show()

In [ ]:
# Distribution by taxonomic class (birds vs amphibians vs mammals etc.)
if "class" in taxonomy_df.columns:
    class_dist = taxonomy_df["class"].value_counts()
    print("Species by taxonomic class:")
    print(class_dist)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    class_dist.plot(kind="bar", ax=ax, color=sns.color_palette("Set2"))
    ax.set_title("Number of species per taxonomic class")
    ax.set_ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Recording Properties

In [ ]:
# Sample recordings to get durations (full scan is slow)
sample_files = train_df.sample(min(200, len(train_df)), random_state=42)["filename"].tolist()

durations = []
for fname in sample_files:
    fpath = os.path.join(TRAIN_AUDIO_DIR, fname)
    if os.path.exists(fpath):
        try:
            dur = librosa.get_duration(path=fpath)
            durations.append(dur)
        except Exception:
            pass

if durations:
    print(f"Sampled {len(durations)} recordings")
    print(f"Duration: min={min(durations):.1f}s, max={max(durations):.1f}s, "
          f"mean={np.mean(durations):.1f}s, median={np.median(durations):.1f}s")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(durations, bins=50, edgecolor="black")
    ax.axvline(5.0, color="red", linestyle="--", label="5s (window size)")
    ax.set_xlabel("Duration (seconds)")
    ax.set_ylabel("Count")
    ax.set_title("Recording duration distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Rating distribution
if "rating" in train_df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    train_df["rating"].hist(bins=20, ax=ax, edgecolor="black")
    ax.axvline(3.0, color="red", linestyle="--", label="min_rating=3.0 threshold")
    ax.set_xlabel("Rating")
    ax.set_ylabel("Count")
    ax.set_title("Recording quality rating distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"Recordings with rating >= 3.0: {(train_df['rating'] >= 3.0).sum()} / {len(train_df)}")

## 4. Geographic Distribution

In [ ]:
if "latitude" in train_df.columns and "longitude" in train_df.columns:
    geo_df = train_df.dropna(subset=["latitude", "longitude"])
    print(f"Recordings with coordinates: {len(geo_df)} / {len(train_df)}")
    
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        geo_df["longitude"], geo_df["latitude"],
        alpha=0.1, s=1, c="blue"
    )
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Recording locations")
    plt.tight_layout()
    plt.show()

## 5. Spectrogram Visualization

In [ ]:
# Visualize sample recordings from different species
audio_config = {
    "sample_rate": 32000,
    "duration": 5.0,
    "n_mels": 128,
    "n_fft": 2048,
    "hop_length": 512,
    "fmin": 50,
    "fmax": 14000,
    "power": 2.0,
    "top_db": 80,
}

# Pick one sample per top-6 species
top_species = train_df["primary_label"].value_counts().head(6).index.tolist()

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for i, species in enumerate(top_species):
    row = train_df[train_df["primary_label"] == species].iloc[0]
    fpath = os.path.join(TRAIN_AUDIO_DIR, row["filename"])
    
    if not os.path.exists(fpath):
        axes[i].set_title(f"{species} - file not found")
        continue
    
    audio = load_audio(fpath, sr=32000, duration=5.0)
    melspec = audio_to_melspec(audio, 32000, audio_config)
    
    librosa.display.specshow(
        melspec, sr=32000, hop_length=512,
        x_axis="time", y_axis="mel",
        ax=axes[i], fmin=50, fmax=14000
    )
    axes[i].set_title(f"{species}")

plt.suptitle("Mel Spectrograms - Sample Species", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Listen to a sample recording
sample_row = train_df.iloc[0]
sample_path = os.path.join(TRAIN_AUDIO_DIR, sample_row["filename"])

if os.path.exists(sample_path):
    audio = load_audio(sample_path, sr=32000, duration=5.0)
    print(f"Species: {sample_row['primary_label']}")
    display(Audio(audio, rate=32000))

## 6. Soundscape Analysis

In [ ]:
# Load soundscape labels
sc_labels_path = os.path.join(DATA_DIR, "train_soundscapes_labels.csv")
if os.path.exists(sc_labels_path):
    sc_labels = pd.read_csv(sc_labels_path)
    print(f"Soundscape annotations: {len(sc_labels)}")
    print(f"Unique files: {sc_labels['filename'].nunique()}")
    print(f"Species: {sc_labels['primary_label'].nunique()}")
    print()
    sc_labels.head(10)

In [ ]:
# Visualize a soundscape with label overlay
if os.path.exists(sc_labels_path):
    sc_files = sorted(os.listdir(TRAIN_SOUNDSCAPES_DIR)) if os.path.exists(TRAIN_SOUNDSCAPES_DIR) else []
    sc_ogg_files = [f for f in sc_files if f.endswith(".ogg")]
    
    if sc_ogg_files:
        sc_file = sc_ogg_files[0]
        sc_path = os.path.join(TRAIN_SOUNDSCAPES_DIR, sc_file)
        
        # Load full soundscape
        audio_full, sr = librosa.load(sc_path, sr=32000, duration=60)
        mel_full = librosa.feature.melspectrogram(
            y=audio_full, sr=sr, n_mels=128, n_fft=2048,
            hop_length=512, fmin=50, fmax=14000
        )
        mel_db = librosa.power_to_db(mel_full, ref=np.max)
        
        fig, ax = plt.subplots(figsize=(18, 5))
        librosa.display.specshow(
            mel_db, sr=sr, hop_length=512,
            x_axis="time", y_axis="mel", ax=ax,
            fmin=50, fmax=14000
        )
        
        # Overlay labels
        file_labels = sc_labels[sc_labels["filename"] == sc_file]
        colors = plt.cm.Set1(np.linspace(0, 1, len(file_labels["primary_label"].unique())))
        species_colors = dict(zip(file_labels["primary_label"].unique(), colors))
        
        for _, row in file_labels.iterrows():
            c = species_colors.get(row["primary_label"], "white")
            ax.axvspan(row["start"], row["end"], alpha=0.2, color=c, label=row["primary_label"])
        
        # Deduplicate legend
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc="upper right", fontsize=8)
        
        ax.set_title(f"Soundscape: {sc_file} (first 60s)")
        plt.tight_layout()
        plt.show()

## 7. Summary

Key findings to inform modeling:
- **Class imbalance**: Long-tail distribution across 234 species - use weighted sampling or focal loss
- **Recording quality**: Filter low-rated recordings (< 3.0) for cleaner training signal
- **Duration**: Most recordings are longer than 5s, so random cropping provides natural augmentation
- **Soundscapes**: Multi-label windows with ambient noise - closer to test distribution